# vLLM Benchmark Results — Full Analysis

**Model:** DeepSeek-R1-Distill-Qwen-7B (FP16) on NVIDIA L4 (24 GB)  
**Engine:** vLLM v0.16+ (V1 engine) · max-model-len 4096 · tensor-parallel 1  
**Load generator:** Locust (streaming SSE) · 90 s per sub-test

## Two test runs

| Run | ID | Dataset | Experiments | Purpose |
|-----|----|---------|-------------|---------|
| **Run 1** | `run_20260306_093145` | `dataset.json` (37 diverse prompts) | 8 configs × 3-4 concurrency levels | Core parameter sweep |
| **Run 2** | `run_20260306_102418` | `prefix_dataset.json` (shared-prefix prompts) | 3 configs × 3-4 concurrency levels | Prefix caching effectiveness |

### Metrics captured
- **Client-side (Locust):** TTFT, TPOT, E2E latency, ITL p50/p99, throughput
- **GPU (nvidia-smi):** utilization %, VRAM, power draw, temperature
- **vLLM (/metrics):** KV cache %, queue depth, prompt/generation throughput

In [ ]:
import glob, json, os, textwrap, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.size': 10,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
})

# ─── Configuration ────────────────────────────────────────────────────────────
RESULTS_ROOT = Path('results')

# Explicit run IDs — set to None to auto-detect
RUN1_ID = 'run_20260306_093145'   # Main parameter sweep (dataset.json)
RUN2_ID = 'run_20260306_102418'   # Prefix caching study  (prefix_dataset.json)

# GPU cost on Lightning.ai L4
GPU_HOURLY_COST_USD = 1.58
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── Helper: load one run's Locust custom-metrics CSVs ────────────────────────
def load_run_locust(run_dir: Path, run_label: str) -> pd.DataFrame:
    """Return a DataFrame with all custom_metrics rows, tagged with experiment / users / run."""
    frames = []
    for csv_path in sorted(run_dir.rglob('*custom_metrics*.csv')):
        parts   = csv_path.stem.split('__')
        exp     = csv_path.parent.name
        users   = int(parts[1].replace('u', '')) if len(parts) > 1 else 0
        cat     = parts[2] if len(parts) > 2 else 'all'
        df = pd.read_csv(csv_path)
        df['experiment'], df['users'], df['prompt_cat_file'], df['run'] = exp, users, cat, run_label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ─── Load BOTH runs ──────────────────────────────────────────────────────────
run1_dir = RESULTS_ROOT / RUN1_ID
run2_dir = RESULTS_ROOT / RUN2_ID

df_run1 = load_run_locust(run1_dir, 'run1')
df_run2 = load_run_locust(run2_dir, 'run2')

df_all = pd.concat([df_run1, df_run2], ignore_index=True)
df_ok  = df_all[df_all['success'] == True].copy()

print(f'Run 1  ({RUN1_ID}): {len(df_run1):>5,} rows  |  {df_run1["experiment"].nunique()} experiments')
print(f'Run 2  ({RUN2_ID}): {len(df_run2):>5,} rows  |  {df_run2["experiment"].nunique()} experiments')
print(f'──────────────────────────────────────────')
print(f'Total loaded   : {len(df_all):>5,}')
print(f'Successful     : {len(df_ok):>5,}  ({100*len(df_ok)/max(len(df_all),1):.1f}%)')
print(f'Failed         : {len(df_all)-len(df_ok):>5,}')

# Split for per-run analysis
r1 = df_ok[df_ok['run'] == 'run1'].copy()
r2 = df_ok[df_ok['run'] == 'run2'].copy()

df_ok.head(3)

In [ ]:
# ─── Experiment matrix overview ───────────────────────────────────────────────
matrix = (
    df_ok.groupby(['run', 'experiment', 'users'])
    .agg(n=('ttft_ms', 'count'))
    .reset_index()
    .pivot_table(index=['run', 'experiment'], columns='users', values='n', fill_value=0)
    .astype(int)
)
matrix.columns = [f'u{c}' for c in matrix.columns]

# Load config.json descriptions
desc_map = {}
for cfg in RESULTS_ROOT.rglob('config.json'):
    c = json.loads(cfg.read_text())
    desc_map[c['name']] = c.get('description', '')

matrix['description'] = [desc_map.get(exp, '') for _, exp in matrix.index]
print('Experiment matrix (request counts per concurrency level):')
display(matrix)

---
# Run 1 — Core Parameter Sweep

8 vLLM configurations tested with diverse prompts (`dataset.json`):
baseline, max_seqs_8, max_seqs_32, gpu_mem_80, gpu_mem_95, chunked_prefill, prefix_caching, chunked_prefill_plus_prefix_cache

In [ ]:
# ─── Percentile summary table (Run 1) ─────────────────────────────────────────
def pct(s, p):
    return s.quantile(p / 100)

def build_summary(df):
    return (
        df.groupby(['experiment', 'users', 'category'])
        .agg(
            n             = ('ttft_ms', 'count'),
            ttft_mean     = ('ttft_ms', 'mean'),
            ttft_p50      = ('ttft_ms', lambda s: pct(s, 50)),
            ttft_p95      = ('ttft_ms', lambda s: pct(s, 95)),
            ttft_p99      = ('ttft_ms', lambda s: pct(s, 99)),
            tpot_mean     = ('tpot_ms', 'mean'),
            tpot_p50      = ('tpot_ms', lambda s: pct(s, 50)),
            tpot_p99      = ('tpot_ms', lambda s: pct(s, 99)),
            e2e_mean      = ('e2e_ms', 'mean'),
            e2e_p50       = ('e2e_ms', lambda s: pct(s, 50)),
            e2e_p99       = ('e2e_ms', lambda s: pct(s, 99)),
            itl_p50_mean  = ('itl_p50_ms', 'mean'),
            itl_p99_mean  = ('itl_p99_ms', 'mean'),
            output_tokens = ('output_tokens', 'sum'),
        )
        .reset_index()
    )

summary1 = build_summary(r1)
summary2 = build_summary(r2)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.1f}'.format)
print(f'Run 1 summary ({len(summary1)} rows):')
display(summary1)

In [ ]:
# ─── TTFT distribution — violin + box by experiment, faceted by concurrency ───
user_levels = sorted(r1['users'].unique())
n_u = len(user_levels)

fig, axes = plt.subplots(1, n_u, figsize=(6 * n_u, 6), sharey=False)
if n_u == 1:
    axes = [axes]

for ax, u in zip(axes, user_levels):
    sub = r1[r1['users'] == u]
    order = sub.groupby('experiment')['ttft_ms'].median().sort_values().index
    sns.violinplot(data=sub, x='experiment', y='ttft_ms', order=order,
                   inner='box', cut=0, ax=ax, palette='Set2', alpha=0.7)
    ax.set_title(f'TTFT — {u} user(s)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('TTFT (ms)')
    ax.tick_params(axis='x', rotation=45)
    # Add median labels
    for i, exp in enumerate(order):
        med = sub[sub['experiment'] == exp]['ttft_ms'].median()
        ax.annotate(f'{med:.0f}', (i, med), textcoords='offset points',
                    xytext=(0, -12), ha='center', fontsize=7, color='black', fontweight='bold')

plt.suptitle('Time to First Token (TTFT) Distribution — Run 1', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(run1_dir / 'ttft_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── TPOT distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, n_u, figsize=(6 * n_u, 6), sharey=False)
if n_u == 1:
    axes = [axes]

for ax, u in zip(axes, user_levels):
    sub = r1[r1['users'] == u]
    order = sub.groupby('experiment')['tpot_ms'].median().sort_values().index
    sns.violinplot(data=sub, x='experiment', y='tpot_ms', order=order,
                   inner='box', cut=0, ax=ax, palette='Set2', alpha=0.7)
    ax.set_title(f'TPOT — {u} user(s)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('TPOT (ms/token)')
    ax.tick_params(axis='x', rotation=45)
    for i, exp in enumerate(order):
        med = sub[sub['experiment'] == exp]['tpot_ms'].median()
        ax.annotate(f'{med:.1f}', (i, med), textcoords='offset points',
                    xytext=(0, -12), ha='center', fontsize=7, fontweight='bold')

plt.suptitle('Time Per Output Token (TPOT) Distribution — Run 1', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(run1_dir / 'tpot_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── E2E Latency distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, n_u, figsize=(6 * n_u, 6), sharey=False)
if n_u == 1:
    axes = [axes]

for ax, u in zip(axes, user_levels):
    sub = r1[r1['users'] == u]
    order = sub.groupby('experiment')['e2e_ms'].median().sort_values().index
    sns.boxplot(data=sub, x='experiment', y='e2e_ms', order=order,
                ax=ax, palette='Set2', flierprops=dict(marker='.', markersize=3, alpha=0.4))
    ax.set_title(f'E2E Latency — {u} user(s)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('E2E Latency (ms)')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('End-to-End Latency Distribution — Run 1', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(run1_dir / 'e2e_boxplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── ITL (Inter-Token Latency) analysis ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Group by experiment + users, compute mean ITL p50 and p99
itl_agg = (
    r1.groupby(['experiment', 'users'])
    .agg(itl_p50=('itl_p50_ms', 'mean'), itl_p99=('itl_p99_ms', 'mean'))
    .reset_index()
)

for ax, metric, title in zip(axes, ['itl_p50', 'itl_p99'], ['ITL P50', 'ITL P99']):
    pivot = itl_agg.pivot(index='experiment', columns='users', values=metric)
    pivot = pivot.reindex(pivot.mean(axis=1).sort_values().index)
    pivot.plot(kind='barh', ax=ax, width=0.75)
    ax.set_xlabel('Inter-Token Latency (ms)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('')
    ax.legend(title='Users', fontsize=8)

plt.suptitle('Inter-Token Latency — Run 1', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(run1_dir / 'itl_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Throughput vs P95 TTFT — Pareto frontier (Run 1) ─────────────────────────
def compute_throughput(grp):
    total_tokens = grp['output_tokens'].sum()
    duration_s = grp['timestamp'].max() - grp['timestamp'].min()
    if duration_s < 1:
        duration_s = 1
    return pd.Series({
        'token_throughput': total_tokens / duration_s,
        'ttft_p95': grp['ttft_ms'].quantile(0.95),
        'ttft_p50': grp['ttft_ms'].quantile(0.50),
    })

pareto = r1.groupby(['experiment', 'users']).apply(compute_throughput).reset_index()

fig, ax = plt.subplots(figsize=(11, 7))
palette = sns.color_palette('tab10', pareto['experiment'].nunique())

for exp, color in zip(sorted(pareto['experiment'].unique()), palette):
    sub = pareto[pareto['experiment'] == exp].sort_values('users')
    ax.plot(sub['ttft_p95'], sub['token_throughput'],
            marker='o', label=exp, color=color, linewidth=2, markersize=7)
    for _, row in sub.iterrows():
        ax.annotate(f'u={int(row["users"])}', (row['ttft_p95'], row['token_throughput']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7, color=color)

ax.set_xlabel('P95 TTFT (ms)  ← lower is better')
ax.set_ylabel('Output Token Throughput (tok/s)  ↑ higher is better')
ax.set_title('Throughput vs Latency — Pareto Frontier (Run 1)', fontweight='bold')
ax.legend(loc='best', fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(run1_dir / 'pareto_frontier.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Cost per million output tokens (Run 1) ───────────────────────────────────
cost_rows = []
for (exp, users), grp in r1.groupby(['experiment', 'users']):
    total_tokens = grp['output_tokens'].sum()
    duration_hr  = (grp['timestamp'].max() - grp['timestamp'].min()) / 3600
    if duration_hr < 1e-6 or total_tokens == 0:
        continue
    cost_per_m = (GPU_HOURLY_COST_USD * duration_hr) / (total_tokens / 1_000_000)
    cost_rows.append({
        'experiment': exp, 'users': users,
        'total_tokens': total_tokens,
        'duration_s': duration_hr * 3600,
        'cost_per_M_tokens': cost_per_m,
    })

cost_df = pd.DataFrame(cost_rows).sort_values(['experiment', 'users'])

fig, ax = plt.subplots(figsize=(10, 5))
for exp in sorted(cost_df['experiment'].unique()):
    sub = cost_df[cost_df['experiment'] == exp]
    ax.plot(sub['users'], sub['cost_per_M_tokens'], marker='o', label=exp, linewidth=1.8)

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('USD per Million Output Tokens')
ax.set_title(f'Cost Efficiency — GPU rate: ${GPU_HOURLY_COST_USD}/hr (Lightning AI L4)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(run1_dir / 'cost_per_token.png', bbox_inches='tight')
plt.show()

display(cost_df.style.format({'cost_per_M_tokens': '${:.2f}', 'duration_s': '{:.0f}s', 'total_tokens': '{:,}'}))

In [ ]:
# ─── Heatmaps: TTFT P50, TTFT P99, TPOT P50 (Run 1) ─────────────────────────
metrics_for_heatmap = [
    ('ttft_p50', 'P50 TTFT (ms)', 'RdYlGn_r'),
    ('ttft_p99', 'P99 TTFT (ms)', 'RdYlGn_r'),
    ('tpot_p50', 'P50 TPOT (ms/tok)', 'RdYlGn_r'),
    ('e2e_p50',  'P50 E2E (ms)', 'RdYlGn_r'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, title, cmap) in zip(axes.flat, metrics_for_heatmap):
    pivot = summary1.pivot_table(index='experiment', columns='users', values=col, aggfunc='mean')
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap=cmap,
                linewidths=0.5, ax=ax, cbar_kws={'label': title})
    ax.set_title(title + ' — lower is better', fontweight='bold', fontsize=11)
    ax.set_xlabel('Concurrent Users')
    ax.set_ylabel('')

plt.suptitle('Latency Heatmaps — Run 1', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(run1_dir / 'latency_heatmaps.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Error analysis (all runs) ────────────────────────────────────────────────
errors = df_all[df_all['success'] == False]
print(f'Total failed requests across all runs: {len(errors)}')

if not errors.empty:
    print('\nError breakdown:')
    display(errors.groupby(['run', 'experiment', 'error_type']).size().rename('count').reset_index())
    fig, ax = plt.subplots(figsize=(8, 4))
    errors.groupby('error_type').size().sort_values().plot(kind='barh', ax=ax, color='salmon')
    ax.set_title('Failed Requests by Error Type')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('✅ No errors in any run — 100% success rate!')

In [ ]:
# ─── Export Run 1 summary to CSV ──────────────────────────────────────────────
summary1.to_csv(run1_dir / 'summary_run1.csv', index=False)
print(f'Run 1 summary saved to: {run1_dir / "summary_run1.csv"}')

---
# Run 2 — Prefix Caching Effectiveness Study

3 experiments using a **shared-prefix dataset** (`prefix_dataset.json`):
- `prefix_baseline` — prefix caching OFF (control)
- `prefix_caching_on` — prefix caching ON only
- `prefix_chunked_plus_cache` — both chunked prefill + prefix caching ON

This isolates the real-world benefit of prefix caching when prompts share common system-prompt prefixes.

In [ ]:
# ─── Run 2: Percentile summary ────────────────────────────────────────────────
print(f'Run 2 summary ({len(summary2)} rows):')
display(summary2)

In [ ]:
# ─── Run 2: Prefix Caching Benefit — TTFT comparison ─────────────────────────
prefix_levels = sorted(r2['users'].unique())
n_p = len(prefix_levels)

fig, axes = plt.subplots(1, n_p, figsize=(6 * n_p, 6), sharey=False)
if n_p == 1:
    axes = [axes]

for ax, u in zip(axes, prefix_levels):
    sub = r2[r2['users'] == u]
    order = sub.groupby('experiment')['ttft_ms'].median().sort_values().index
    sns.violinplot(data=sub, x='experiment', y='ttft_ms', order=order,
                   inner='box', cut=0, ax=ax, palette='Pastel1', alpha=0.8)
    ax.set_title(f'TTFT — {u} user(s)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('TTFT (ms)')
    ax.tick_params(axis='x', rotation=30)
    for i, exp in enumerate(order):
        med = sub[sub['experiment'] == exp]['ttft_ms'].median()
        ax.annotate(f'{med:.0f}', (i, med), textcoords='offset points',
                    xytext=(0, -12), ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Prefix Caching — TTFT Comparison (Shared-Prefix Dataset)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(run2_dir / 'prefix_ttft_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Run 2: Prefix Caching Benefit — percentage improvement table ─────────────
# Compare each prefix config against prefix_baseline
baseline_key = 'prefix_baseline'
if baseline_key in r2['experiment'].unique():
    base = summary2[summary2['experiment'] == baseline_key].set_index('users')
    
    improvement_rows = []
    for exp in summary2['experiment'].unique():
        if exp == baseline_key:
            continue
        comp = summary2[summary2['experiment'] == exp].set_index('users')
        for u in comp.index.intersection(base.index):
            b, c = base.loc[u], comp.loc[u]
            improvement_rows.append({
                'experiment': exp,
                'users': u,
                'ttft_p50_change_%': (c['ttft_p50'] - b['ttft_p50']) / b['ttft_p50'] * 100,
                'ttft_p99_change_%': (c['ttft_p99'] - b['ttft_p99']) / b['ttft_p99'] * 100,
                'tpot_p50_change_%': (c['tpot_p50'] - b['tpot_p50']) / b['tpot_p50'] * 100,
                'e2e_p50_change_%':  (c['e2e_p50'] - b['e2e_p50']) / b['e2e_p50'] * 100,
            })
    
    imp_df = pd.DataFrame(improvement_rows)
    print('Percentage change vs prefix_baseline (negative = improvement):')
    display(imp_df.style.format('{:+.1f}%').map(
        lambda v: 'color: green' if isinstance(v, str) and v.startswith('-') else 'color: red' if isinstance(v, str) and v.startswith('+') else ''
    ))

    # Bar chart of TTFT P50 improvement
    fig, ax = plt.subplots(figsize=(8, 4))
    for exp in imp_df['experiment'].unique():
        sub = imp_df[imp_df['experiment'] == exp]
        ax.bar([f'u={u}' for u in sub['users']], sub['ttft_p50_change_%'], label=exp, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('TTFT P50 Change (%)')
    ax.set_title('Prefix Caching Impact on TTFT P50', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(run2_dir / 'prefix_improvement.png', bbox_inches='tight')
    plt.show()
else:
    print(f'⚠ No "{baseline_key}" experiment found in Run 2')

# Export Run 2 summary
summary2.to_csv(run2_dir / 'summary_run2.csv', index=False)
print(f'Run 2 summary saved to: {run2_dir / "summary_run2.csv"}')

---
# GPU & vLLM Server-Side Metrics

The following cells load `*_gpu_metrics.csv` and `*_vllm_metrics.csv` files that were captured by `gpu_monitor.sh` alongside each Locust test. These show **how hard the GPU was working** during each experiment.

In [ ]:
# ─── Load GPU metrics (nvidia-smi) — both runs ────────────────────────────────
def load_gpu(run_dir: Path, run_label: str) -> list:
    frames = []
    for csv_path in sorted(run_dir.rglob('*_gpu_metrics.csv')):
        parts = csv_path.stem.replace('_gpu_metrics', '').split('__')
        exp   = csv_path.parent.name
        users = int(parts[1].replace('u', '')) if len(parts) > 1 else 0
        try:
            gdf = pd.read_csv(csv_path)
            gdf['timestamp'] = pd.to_datetime(gdf['timestamp'], errors='coerce')
            gdf['experiment'], gdf['users'], gdf['run'] = exp, users, run_label
            gdf['elapsed_s'] = (gdf['timestamp'] - gdf['timestamp'].min()).dt.total_seconds()
            frames.append(gdf)
        except Exception as e:
            print(f'Warning: {csv_path.name}: {e}')
    return frames

gpu_frames = load_gpu(run1_dir, 'run1') + load_gpu(run2_dir, 'run2')
df_gpu = pd.concat(gpu_frames, ignore_index=True) if gpu_frames else pd.DataFrame()
print(f'GPU metrics: {len(df_gpu):,} samples, {df_gpu["experiment"].nunique() if not df_gpu.empty else 0} experiments')

In [ ]:
# ─── Load vLLM metrics (/metrics endpoint) — both runs ────────────────────────
def load_vllm(run_dir: Path, run_label: str) -> list:
    frames = []
    for csv_path in sorted(run_dir.rglob('*_vllm_metrics.csv')):
        parts = csv_path.stem.replace('_vllm_metrics', '').split('__')
        exp   = csv_path.parent.name
        users = int(parts[1].replace('u', '')) if len(parts) > 1 else 0
        try:
            vdf = pd.read_csv(csv_path)
            vdf['timestamp'] = pd.to_datetime(vdf['timestamp'], errors='coerce')
            vdf['experiment'], vdf['users'], vdf['run'] = exp, users, run_label
            vdf['elapsed_s'] = (vdf['timestamp'] - vdf['timestamp'].min()).dt.total_seconds()
            frames.append(vdf)
        except Exception as e:
            print(f'Warning: {csv_path.name}: {e}')
    return frames

vllm_frames = load_vllm(run1_dir, 'run1') + load_vllm(run2_dir, 'run2')
df_vllm = pd.concat(vllm_frames, ignore_index=True) if vllm_frames else pd.DataFrame()
print(f'vLLM metrics: {len(df_vllm):,} samples, {df_vllm["experiment"].nunique() if not df_vllm.empty else 0} experiments')

In [ ]:
# ─── GPU Utilization & VRAM Time Series (per experiment × concurrency) ────────
if not df_gpu.empty:
    combos = df_gpu.groupby(['experiment', 'users']).ngroups
    n_cols = min(3, combos)
    n_rows = (combos + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, ((exp, u), grp) in enumerate(df_gpu.groupby(['experiment', 'users'])):
        if idx >= len(axes_flat):
            break
        ax = axes_flat[idx]
        ax2 = ax.twinx()

        ax.plot(grp['elapsed_s'], grp['gpu_utilization_pct'], color='tab:blue', alpha=0.8, label='GPU Util %')
        ax2.plot(grp['elapsed_s'], grp['memory_used_mb'], color='tab:red', alpha=0.8, label='VRAM (MB)')

        ax.set_ylim(0, 105)
        ax.set_ylabel('GPU Utilization (%)', color='tab:blue')
        ax2.set_ylabel('VRAM Used (MB)', color='tab:red')
        ax.set_xlabel('Elapsed (s)')
        ax.set_title(f'{exp} — {u} user(s)', fontsize=10)
        ax.legend(loc='upper left', fontsize=7)
        ax2.legend(loc='upper right', fontsize=7)

    # Hide unused axes
    for idx_ax in range(idx + 1, len(axes_flat)):
        axes_flat[idx_ax].set_visible(False)

    fig.suptitle('GPU Utilization & VRAM Usage Over Time', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(run_dir / 'gpu_utilization_timeseries.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ Skipping GPU time series — no GPU metrics available.')

In [ ]:
# ─── KV Cache Utilization & Queue Depth Time Series ──────────────────────────
if not df_vllm.empty:
    combos = df_vllm.groupby(['experiment', 'users']).ngroups
    n_cols = min(3, combos)
    n_rows = (combos + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, ((exp, u), grp) in enumerate(df_vllm.groupby(['experiment', 'users'])):
        if idx >= len(axes_flat):
            break
        ax = axes_flat[idx]
        ax2 = ax.twinx()

        ax.plot(grp['elapsed_s'], grp['gpu_cache_usage_pct'], color='tab:green', alpha=0.8, label='KV Cache %')
        ax2.plot(grp['elapsed_s'], grp['num_requests_running'], color='tab:orange', alpha=0.8, label='Running')
        ax2.plot(grp['elapsed_s'], grp['num_requests_waiting'], color='tab:red', alpha=0.6, linestyle='--', label='Waiting')

        ax.set_ylim(0, 105)
        ax.set_ylabel('KV Cache Usage (%)', color='tab:green')
        ax2.set_ylabel('Request Count', color='tab:orange')
        ax.set_xlabel('Elapsed (s)')
        ax.set_title(f'{exp} — {u} user(s)', fontsize=10)
        ax.legend(loc='upper left', fontsize=7)
        ax2.legend(loc='upper right', fontsize=7)

    for idx_ax in range(idx + 1, len(axes_flat)):
        axes_flat[idx_ax].set_visible(False)

    fig.suptitle('KV Cache Utilization & Queue Depth Over Time', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(run_dir / 'kv_cache_timeseries.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ Skipping KV cache time series — no vLLM metrics available.')

In [ ]:
# ─── GPU Power & Temperature Time Series ─────────────────────────────────────
if not df_gpu.empty:
    combos = df_gpu.groupby(['experiment', 'users']).ngroups
    n_cols = min(3, combos)
    n_rows = (combos + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, ((exp, u), grp) in enumerate(df_gpu.groupby(['experiment', 'users'])):
        if idx >= len(axes_flat):
            break
        ax = axes_flat[idx]
        ax2 = ax.twinx()

        ax.plot(grp['elapsed_s'], grp['power_draw_w'], color='tab:purple', alpha=0.8, label='Power (W)')
        ax2.plot(grp['elapsed_s'], grp['temperature_c'], color='tab:brown', alpha=0.8, label='Temp (°C)')

        ax.set_ylabel('Power Draw (W)', color='tab:purple')
        ax2.set_ylabel('Temperature (°C)', color='tab:brown')
        ax.set_xlabel('Elapsed (s)')
        ax.set_title(f'{exp} — {u} user(s)', fontsize=10)
        ax.legend(loc='upper left', fontsize=7)
        ax2.legend(loc='upper right', fontsize=7)

    for idx_ax in range(idx + 1, len(axes_flat)):
        axes_flat[idx_ax].set_visible(False)

    fig.suptitle('GPU Power Draw & Temperature Over Time', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(run_dir / 'gpu_power_temp_timeseries.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ Skipping power/temp time series — no GPU metrics available.')

## Cross-Experiment GPU Stress Comparison

Aggregate GPU metrics across experiments to see which configurations push the GPU hardest.

In [ ]:
# ─── GPU Stress Summary Table ─────────────────────────────────────────────────
if not df_gpu.empty:
    gpu_summary = (
        df_gpu.groupby(['experiment', 'users'])
        .agg(
            gpu_util_mean   = ('gpu_utilization_pct', 'mean'),
            gpu_util_max    = ('gpu_utilization_pct', 'max'),
            vram_mean_mb    = ('memory_used_mb', 'mean'),
            vram_max_mb     = ('memory_used_mb', 'max'),
            vram_total_mb   = ('memory_total_mb', 'first'),
            power_mean_w    = ('power_draw_w', 'mean'),
            power_max_w     = ('power_draw_w', 'max'),
            temp_mean_c     = ('temperature_c', 'mean'),
            temp_max_c      = ('temperature_c', 'max'),
        )
        .reset_index()
    )
    gpu_summary['vram_util_mean_pct'] = (gpu_summary['vram_mean_mb'] / gpu_summary['vram_total_mb'] * 100).round(1)
    
    pd.set_option('display.float_format', '{:.1f}'.format)
    display(gpu_summary)
else:
    print('⚠ No GPU metrics to summarize.')

In [ ]:
# ─── KV Cache Stress Summary & Heatmap ────────────────────────────────────────
if not df_vllm.empty:
    vllm_summary = (
        df_vllm.groupby(['experiment', 'users'])
        .agg(
            kv_cache_mean_pct  = ('gpu_cache_usage_pct', 'mean'),
            kv_cache_max_pct   = ('gpu_cache_usage_pct', 'max'),
            requests_running_mean = ('num_requests_running', 'mean'),
            requests_waiting_mean = ('num_requests_waiting', 'mean'),
            requests_waiting_max  = ('num_requests_waiting', 'max'),
            gen_throughput_mean   = ('avg_generation_throughput_toks_per_s', 'mean'),
        )
        .reset_index()
    )
    display(vllm_summary)

    # Heatmap: mean KV cache usage
    pivot_kv = vllm_summary.pivot_table(
        index='experiment', columns='users', values='kv_cache_mean_pct', aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(max(6, pivot_kv.shape[1] * 1.5), max(4, pivot_kv.shape[0] * 0.7)))
    sns.heatmap(pivot_kv, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'KV Cache Usage (%)'})
    ax.set_title('Mean KV Cache Utilization (%) — higher = more memory pressure')
    ax.set_xlabel('Concurrent Users')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(run_dir / 'heatmap_kv_cache.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ No vLLM metrics to summarize.')

In [ ]:
# ─── Grouped Bar: Avg GPU Util by experiment × concurrency ────────────────────
if not df_gpu.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    gpu_pivot = gpu_summary.pivot(index='experiment', columns='users', values='gpu_util_mean')
    gpu_pivot.plot(kind='bar', ax=ax, width=0.75)
    ax.set_ylabel('Mean GPU Utilization (%)')
    ax.set_xlabel('')
    ax.set_title('Mean GPU Utilization by Experiment & Concurrency')
    ax.legend(title='Users', fontsize=8)
    ax.set_ylim(0, 105)
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.savefig(run_dir / 'gpu_util_bar_chart.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ─── Unified Summary: Merge Locust + GPU + vLLM metrics ──────────────────────
# Join all three data sources by (experiment, users) for a single overview table

unified = summary.groupby(['experiment', 'users']).agg(
    n          = ('n', 'sum'),
    ttft_mean  = ('ttft_mean', 'mean'),
    ttft_p99   = ('ttft_p99', 'max'),
    tpot_mean  = ('tpot_mean', 'mean'),
    e2e_mean   = ('e2e_mean', 'mean'),
).reset_index()

if not df_gpu.empty:
    unified = unified.merge(
        gpu_summary[['experiment', 'users', 'gpu_util_mean', 'vram_mean_mb', 'vram_util_mean_pct', 'power_mean_w', 'temp_mean_c']],
        on=['experiment', 'users'], how='left'
    )

if not df_vllm.empty:
    unified = unified.merge(
        vllm_summary[['experiment', 'users', 'kv_cache_mean_pct', 'requests_waiting_mean', 'gen_throughput_mean']],
        on=['experiment', 'users'], how='left'
    )

# Save unified results
unified_path = run_dir / 'unified_summary.csv'
unified.to_csv(unified_path, index=False)
print(f'Unified summary saved to: {unified_path}')
display(unified)

In [ ]:
# ─── Correlation: KV Cache vs TTFT ────────────────────────────────────────────
if not df_vllm.empty and 'kv_cache_mean_pct' in unified.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # KV Cache vs TTFT P99
    for exp in unified['experiment'].unique():
        sub = unified[unified['experiment'] == exp]
        ax1.scatter(sub['kv_cache_mean_pct'], sub['ttft_p99'], label=exp, s=60)
        for _, row in sub.iterrows():
            ax1.annotate(f'u={int(row["users"])}', (row['kv_cache_mean_pct'], row['ttft_p99']),
                        textcoords='offset points', xytext=(4, 4), fontsize=7)
    ax1.set_xlabel('Mean KV Cache Usage (%)')
    ax1.set_ylabel('P99 TTFT (ms)')
    ax1.set_title('KV Cache Pressure → TTFT Impact')
    ax1.legend(fontsize=7)

    # GPU Util vs Generation Throughput
    if 'gpu_util_mean' in unified.columns and 'gen_throughput_mean' in unified.columns:
        for exp in unified['experiment'].unique():
            sub = unified[unified['experiment'] == exp]
            ax2.scatter(sub['gpu_util_mean'], sub['gen_throughput_mean'], label=exp, s=60)
            for _, row in sub.iterrows():
                ax2.annotate(f'u={int(row["users"])}', (row['gpu_util_mean'], row['gen_throughput_mean']),
                            textcoords='offset points', xytext=(4, 4), fontsize=7)
        ax2.set_xlabel('Mean GPU Utilization (%)')
        ax2.set_ylabel('Avg Generation Throughput (tok/s)')
        ax2.set_title('GPU Utilization → Throughput')
        ax2.legend(fontsize=7)

    plt.tight_layout()
    plt.savefig(run_dir / 'correlation_plots.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ Skipping correlation analysis — missing GPU/vLLM metrics.')